# Repertoire Overlap Correctness Check

[03_airr_overlap.ipynb](03_airr_overlap.ipynb) benchmarks the *runtime* of four repertoire-overlap implementations (SymDel, SymScan, XTNeighbor-streaming and CompAIRR), but never checks that they agree on the *result*. This notebook closes that gap: it runs all four implementations on small samples of real repertoires (from [Emerson et al](https://doi.org/10.1038/ng.3822)) and checks that the resulting repertoire x repertoire overlap matrices are identical, using CompAIRR as the reference.

Note: CompAIRR only supports indels (Levenshtein distance) when `d=1` (`d=2` errors out), so it is excluded from the `d=2` Levenshtein comparison; SymDel, SymScan and XTNeighbor-streaming are compared directly against each other in that case instead.

The notebook mirrors the structure of `03_airr_overlap.ipynb`:
1. __Configuration__
2. __Setup:__ install dependencies and compile the implementations
3. __Implementations:__ standardize every algorithm to return an overlap matrix
4. __Correctness Check:__ run every algorithm on small repertoire samples and compare outputs

Run time: a few minutes, much shorter than the full benchmark since only small repertoire samples are needed to establish correctness.

## 0. Configuration

In [ ]:
n_repeat = 1 # number of random repertoire subsets to test per configuration
sizes = [1, 2, 4, 8] # number of repertoires to sample
max_seqs_per_repertoire = 3000 # subsample each repertoire down to this many sequences, to keep CompAIRR's d=2 runs fast

## 1. Setup (run time ~ 3 min)

install dependency

In [ ]:
! pip install -q pyrepseq

In [ ]:
import os.path
import numpy as np
import pandas as pd

from airrutils import *

try:
    from google.colab import files
    colab = True
except ImportError:
    colab = False

clone the projects

In [ ]:
if not os.path.exists("compairr"):
    !git clone https://github.com/uio-bmi/compairr.git

if colab and not os.path.exists("XT-neighbor"):
    !git clone https://github.com/heartnetkung/XT-neighbor.git
    repo_path = "XT-neighbor/"
else:
    repo_path = "../"


compile XTNeighbor-streaming

In [ ]:
! mkdir -p {repo_path}xtneighbor_streaming/build
! cd {repo_path}xtneighbor_streaming/build; cmake ..;make

compile Compairr

In [ ]:
!cd compairr; make

prepare repertoire info

In [ ]:
def read_info():
  ans = pd.read_csv(f'{repo_path}/data/info.csv')
  end = np.cumsum(ans['count'])
  ans['start'] = np.concatenate(([0],end[:-1]))
  ans['end'] = end
  return ans

info = read_info()
info

In [ ]:
! mkdir -p tmp

prepare input data

In [ ]:
N_FILES=5

def read_input():
  for i in range(1,N_FILES+1):
    ! unzip -n {repo_path}/data/emerson_rep"$i".zip -d tmp
  reps = []
  for i in range(1,N_FILES+1):
    reps.append(pd.read_csv(f'tmp/emerson_rep{i}.txt'))
  return pd.concat(reps,ignore_index=True)

data = read_input()
print(data.head())

check GPU availability

In [ ]:
import subprocess
try:
  subprocess.run(["nvidia-smi"], capture_output=True, text=True)
except Exception as e:
  raise Exception("GPU required")

input preparation code, adapted from 03_airr_overlap.ipynb to subsample each repertoire down to `max_seqs_per_repertoire` sequences, since correctness only requires a modest number of matching pairs, not the full repertoire size

## 2. Correctness Check (run time ~ a few minutes)

comparison harness: run every algorithm and compare its matrix against CompAIRR (or, when CompAIRR is unsupported, against whichever algorithm ran first)

In [ ]:
algorithms = {
    'symdel': symdel_overlap,
    'symscan': symscan_overlap,
    'xt_streaming': lambda *a, **kw: xt_neighbor_overlap(*a, **kw, return_matrix=True),
    'compairr': lambda *a, **kw: compairr_overlap(*a, **kw, return_matrix=True),
}

def compare(distance, is_hamming, seqs, dup_counts, rep_sizes):
  matrices = {}
  for alg_name, fn in algorithms.items():
    result = fn(distance, is_hamming, seqs, dup_counts, rep_sizes)
    if result is not None:
      matrices[alg_name] = result

  ref_name = 'compairr' if 'compairr' in matrices else next(iter(matrices))
  ref = matrices[ref_name]

  mismatches = []
  for alg_name, mat in matrices.items():
    if alg_name == ref_name:
      continue
    ok = np.array_equal(mat, ref)
    print(f'    {alg_name:15s} vs {ref_name}: {"MATCH" if ok else "MISMATCH"}')
    if not ok:
      mismatches.append(alg_name)
  return mismatches


def run_exp(distance, is_hamming):
  measure = 'hamming' if is_hamming else 'leven'
  mismatches = set()
  for i in range(n_repeat):
    for size in sizes:
      seq_info, info_subset = sample_repertoire(data, info, size, random_state=i, max_seqs=max_seqs_per_repertoire)
      seqs, dup_counts, rep_sizes = prepare(seq_info, info_subset)
      print(f'distance={distance} measure={measure} n_repertoire={size} repeat={i}')
      mismatches.update(compare(distance, is_hamming, seqs, dup_counts, rep_sizes))
  return mismatches

In [ ]:
mismatches = set()

In [ ]:
mismatches |= run_exp(distance=1, is_hamming=True)

In [ ]:
mismatches |= run_exp(distance=1, is_hamming=False)

In [ ]:
mismatches |= run_exp(distance=2, is_hamming=True)

In [ ]:
mismatches |= run_exp(distance=2, is_hamming=False)

In [ ]:
if mismatches:
    raise Exception(f'comparison failed for: {sorted(mismatches)}')
print('success!')